# 9장 - 로컬 Whisper STT

원본 파일: `chap09/whisper_local.py`

In [1]:
import os
import torch
import pandas as pd
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

BASE_DIR = (os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd())

/Users/seminy/Desktop/Main/Git/AI_Bootcamp/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/seminy/Desktop/Main/Git/AI_Bootcamp/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def whisper_to_dataframe(result, output_file_path):
    """위스퍼 결과(청크별 타임스탬프+텍스트)를 표(DataFrame)로 정리하고 CSV로 저장"""
    start_end_text = []

    for chunk in result["chunks"]:
        start = chunk["timestamp"][0]
        end = chunk["timestamp"][1]
        text = chunk["text"].strip()
        start_end_text.append([start, end, text])

    df = pd.DataFrame(start_end_text, columns=["start", "end", "text"])
    df.to_csv(output_file_path, index=False, sep="|")
    return df

In [3]:
def whisper_stt(audio_file_path: str, output_file_path: str = "./output.csv"):
    """로컬에 다운로드한 위스퍼 모델로 음성 파일을 텍스트로 변환"""
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model_id = "openai/whisper-tiny"  # 가벼운 모델로 CPU에서도 빠르게 테스트 가능

    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        model_id, torch_dtype=torch_dtype,
        low_cpu_mem_usage=True,
        use_safetensors=True,
    )
    model.to(device)

    processor = AutoProcessor.from_pretrained(model_id)

    pipe = pipeline(
        "automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        torch_dtype=torch_dtype,
        device=device,
        return_timestamps=True,  # 청크별로 타임스탬프 반환
        chunk_length_s=10,       # 입력 오디오 10초씩 나누기
        stride_length_s=2,       # 2초씩 겹치도록 청크 나누기
    )

    result = pipe(audio_file_path)
    df = whisper_to_dataframe(result, output_file_path)

    return result, df

## 실행 / 테스트

In [4]:
audio_path = os.path.join(BASE_DIR, 'data', 'lsy_audio_2023_58s.mp3')
output_path = os.path.join(BASE_DIR, 'data', 'lsy_audio_2023_58s.csv')

In [5]:
result, df = whisper_stt(audio_path, output_path)
print(df)

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cpu
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to E

   start    end                                               text
0   0.00   6.70       안녕하세요, 강인은 GBT API로 첫 번 만들기 라는 내용을 다루는 강입니다.
1   6.70  11.00                      GBT API에 대해서 뭐 생소하신 분들도 있을 텐데
2  11.00  17.08           우리가 잘 알고 있는 채취빛이 채취빛빛을 이용을 최집빛의 기능을 이용해서
3  17.08  35.24  우리가 원하는 프로그램 어떻게 만드는지에 대해서 이야기할 거예요. 그래서 이런 강의...
4  35.24  48.70  그리고 그걸 왜 필요한지에 대해서 좀 이야기를 할 거고요 그 예재로 예재는 여러 가...
5  49.70  58.00                           그래서 프로그램을 실� 민제에 나타나게 되고
